# QSAR explainability comparison (SHAP and BorutaShap)

This notebook compares explainability outputs from RF and XGB by combining SHAP and BorutaShap values into one feature-level table. It also exports target-level Spearman correlation summaries for cross-method and cross-model alignment.

It requires pre-generated explainability checkpoints (SHAP `.npy` files and BorutaShap score `.pkl` files) under the expected `checkpoints/` directories. Generate them first by running: `qsar/random-forest/random-forest-shap.ipynb`, `qsar/xgboost/xgboost-shap.ipynb`, `qsar/random-forest/random-forest-boruta-shap.ipynb`, and `qsar/xgboost/xgboost-boruta-shap.ipynb`.

In [1]:
from pathlib import Path
import pickle
import sys

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

cwd = Path.cwd().resolve()
if cwd.name == "qsar":
    base_dir = cwd
elif (cwd / "qsar").exists():
    base_dir = (cwd / "qsar").resolve()
else:
    base_dir = cwd

if str(base_dir) not in sys.path:
    sys.path.insert(0, str(base_dir))

from qsar_common import (
    add_mol_column,
    aggregate_targets_by_fingerprint,
    build_morgan_fingerprints,
    encode_targets,
    load_qsar_dataset,
)
from qsar_config import DATA_PATH, ECFP_N_BITS, ECFP_RADIUS

In [2]:
def target_names_list():
    df = load_qsar_dataset(DATA_PATH)
    df = add_mol_column(df, smiles_column="Smiles", mol_column="mol")
    df = build_morgan_fingerprints(
        df,
        mol_column="mol",
        output_column="fp",
        radius=ECFP_RADIUS,
        n_bits=ECFP_N_BITS,
    )
    df, _, target_names = encode_targets(
        df,
        target_column="Target",
        output_column="target_encoded",
    )
    _ = aggregate_targets_by_fingerprint(
        df,
        fp_column="fp",
        encoded_target_column="target_encoded",
        aggregated_target_column="target",
    )
    return [str(t) for t in target_names]


def load_npy(path, n_bits):
    if not path.exists():
        return np.full(n_bits, np.nan, dtype=float)
    arr = np.asarray(np.load(path), dtype=float).reshape(-1)
    if arr.shape[0] != n_bits:
        out = np.full(n_bits, np.nan, dtype=float)
        out[: min(n_bits, arr.shape[0])] = arr[: min(n_bits, arr.shape[0])]
        return out
    return arr


def load_boruta_scores(path, n_bits):
    expected_index = pd.Index([f"bit_{i}" for i in range(n_bits)])
    if not path.exists():
        return pd.Series(np.full(n_bits, np.nan), index=expected_index)

    with open(path, "rb") as f:
        cache_obj = pickle.load(f)

    data = cache_obj.get("data", {})
    mean_z = data.get("mean_z_scores")
    if mean_z is None:
        return pd.Series(np.full(n_bits, np.nan), index=expected_index)

    if isinstance(mean_z, pd.Series):
        series = mean_z.astype(float)
    else:
        arr = np.asarray(mean_z, dtype=float).reshape(-1)
        series = pd.Series(arr, index=[f"bit_{i}" for i in range(arr.shape[0])])

    return series.reindex(expected_index)


def rank_abs(series):
    return series.abs().rank(method="average", ascending=False)


def safe_spearman(a, b):
    mask = a.notna() & b.notna()
    if mask.sum() < 3:
        return np.nan
    corr, _ = spearmanr(a[mask], b[mask])
    return float(corr) if corr == corr else np.nan

In [ ]:
n_bits = ECFP_N_BITS

rf_shap_dir = base_dir / "random-forest" / "checkpoints" / "random-forest-shap"
xgb_shap_dir = base_dir / "xgboost" / "checkpoints" / "xgboost-shap"
rf_boruta_dir = base_dir / "random-forest" / "checkpoints" / "boruta-shap"
xgb_boruta_dir = base_dir / "xgboost" / "checkpoints" / "boruta-shap"

required_dirs = {
    "RF SHAP": rf_shap_dir,
    "XGB SHAP": xgb_shap_dir,
    "RF BorutaShap": rf_boruta_dir,
    "XGB BorutaShap": xgb_boruta_dir,
}
missing_dirs = [f"{name}: {path}" for name, path in required_dirs.items() if not path.exists()]
if missing_dirs:
    missing_block = "\n".join(f"- {entry}" for entry in missing_dirs)
    raise FileNotFoundError(
        "Explainability checkpoints are not generated yet.\n"
        "Generate SHAP and BorutaShap checkpoints first, then rerun this notebook.\n"
        "Missing checkpoint directories:\n"
        f"{missing_block}"
    )

all_rows = []
summary_rows = []

for target in target_names_list():
    safe_target = target.replace("/", "_").replace(" ", "_")

    target_required_files = {
        "RF SHAP": rf_shap_dir / f"target_{target}.npy",
        "XGB SHAP": xgb_shap_dir / f"target_{target}.npy",
        "RF BorutaShap": rf_boruta_dir / f"target_{safe_target}_scores.pkl",
        "XGB BorutaShap": xgb_boruta_dir / f"target_{safe_target}_scores.pkl",
    }
    missing_files = [
        f"{name}: {path}"
        for name, path in target_required_files.items()
        if not path.exists()
    ]
    if missing_files:
        missing_block = "\n".join(f"- {entry}" for entry in missing_files)
        raise FileNotFoundError(
            f"Checkpoint files are missing for target '{target}'.\n"
            "Generate explainability checkpoints before running this comparison notebook.\n"
            "Missing checkpoint files:\n"
            f"{missing_block}"
        )

    rf_shap = load_npy(rf_shap_dir / f"target_{target}.npy", n_bits)
    xgb_shap = load_npy(xgb_shap_dir / f"target_{target}.npy", n_bits)

    rf_boruta = load_boruta_scores(rf_boruta_dir / f"target_{safe_target}_scores.pkl", n_bits)
    xgb_boruta = load_boruta_scores(xgb_boruta_dir / f"target_{safe_target}_scores.pkl", n_bits)

    bit_idx = np.arange(n_bits)
    frame = pd.DataFrame(
        {
            "target": target,
            "bit_idx": bit_idx,
            "bit_name": [f"bit_{i}" for i in bit_idx],
            "rf_shap": rf_shap,
            "xgb_shap": xgb_shap,
            "rf_boruta_z": rf_boruta.values,
            "xgb_boruta_z": xgb_boruta.values,
        }
    )

    frame["rf_shap_rank"] = rank_abs(frame["rf_shap"])
    frame["xgb_shap_rank"] = rank_abs(frame["xgb_shap"])
    frame["rf_boruta_rank"] = rank_abs(frame["rf_boruta_z"])
    frame["xgb_boruta_rank"] = rank_abs(frame["xgb_boruta_z"])

    all_rows.append(frame)

    summary_rows.append(
        {
            "target": target,
            "spearman_rf_shap_vs_rf_boruta": safe_spearman(frame["rf_shap"], frame["rf_boruta_z"]),
            "spearman_xgb_shap_vs_xgb_boruta": safe_spearman(frame["xgb_shap"], frame["xgb_boruta_z"]),
            "spearman_rf_shap_vs_xgb_shap": safe_spearman(frame["rf_shap"], frame["xgb_shap"]),
            "spearman_rf_boruta_vs_xgb_boruta": safe_spearman(frame["rf_boruta_z"], frame["xgb_boruta_z"]),
        }
    )

combined_df = pd.concat(all_rows, ignore_index=True)
summary_df = pd.DataFrame(summary_rows)

out_dir = base_dir / "outputs" / "model-comparison"
out_dir.mkdir(parents=True, exist_ok=True)

combined_path = out_dir / "explainability_feature_comparison.csv"
summary_path = out_dir / "explainability_target_summary.csv"
combined_df.to_csv(combined_path, index=False)
summary_df.to_csv(summary_path, index=False)

print(f"Saved feature-level explainability comparison: {combined_path}")
print(f"Saved target-level explainability summary: {summary_path}")
summary_df

Saved feature-level explainability comparison: /mnt/ArchData/GitHub/qspr-qsar-explainability/qsar/outputs/model-comparison/explainability_feature_comparison.csv
Saved target-level explainability summary: /mnt/ArchData/GitHub/qspr-qsar-explainability/qsar/outputs/model-comparison/explainability_target_summary.csv


,target,spearman_rf_shap_vs_rf_boruta,spearman_xgb_shap_vs_xgb_boruta,spearman_rf_shap_vs_xgb_shap,spearman_rf_boruta_vs_xgb_boruta
0,ar,0.750734,0.675129,0.460412,0.531896
1,era,0.763274,0.688462,0.480825,0.533657
2,erb,0.764144,0.706552,0.486753,0.548622
3,gr,0.758546,0.669853,0.457684,0.546395
4,mr,0.730967,0.673889,0.448040,0.514055
5,pr,0.764823,0.708135,0.486607,0.552082
